### React
- Reasion(추론)
- Acting(행동)
- 두개를 결합한 LLM 에이전트

- 1.사고
- 2.행동
- 3.관찰
- 4.재추론
을 반복하면서 문제를 해결하도록 만든 구조

### Prompting 한계
- 질문 -> 답변
    - 환각, 최신정보가 부족, 외부시스템 접근 불가, 추론한계
    ### Chain of Thought(Cot)
        - 질문 -단계별 사고-답변
        - 검색/api 호출 불가
        - 정확도 부족
### ReAct 구조
- 루프반복
    - Action
    - Observation
    - Thought
    - Action
    - Observation
    - Thought    
    - Final Answer

    - 생각
    - 도구사용
    - 결과를 관찰
    - 다시생각

- Zero-shot --> Few-shot --> Chain-of_Thought --> ReAct

In [1]:
# 프롬프팅 시뮬레이션
question = '대한민국의 수도는 어디이며, 그 도시의 인구는 얼마인가?'

# Zero-shot  예시나 질문없이 바로 질문
zero_shot_prompt = f'''질문:{question}\n답변:'''
few_shot_prompt = f'''질문:일본의 수도는 어디이며, 그 도시의 인구는 얼마인가
답변: 일본의 수도는 도쿄이며,인구는 약 1,500만 명입니다.
질문:{question}\n답변:'''
cot_prompt = f'''질문:{question}\n단계별로 생각해보겠습니다.\n1단계: 대한민국의 수도를 확인합니다.
2단계: 해당 도시의 인구 데이터를 조회합니다.\n답변:'''
# 추론과 행동을 교차 반복
react_prompt = f'''질문:{question}
Thought:이 질문에 답하려면 두 가지 정보가 필요합니다. 먼저 대한민국의 수도를 확인하고, 그 다음 해당도시의 인구를 조회해야합니다.
Action : Search[대한민국수도]
Observation: 대한민국의 수도는 서울특별시입니다.
Thought: 서울이 대한민국의 수도인것을 확인했습니다. 이제 서울의인구를 조회해야 합니다.
Action : Search[서울특별시 인구]
Observation:서울특별시의 인구는 약 900만명입니다.
Thought:두 가지 정보를 모두 확인했으므로 최종 답변을 구성합니다
Action: Finish[대한민국의 수도는 서울이며, 서울의 인구는 약 900만 명입니다.]
'''

In [2]:
# React 루프구조 시뮬레이션
class ReActStep:
    '''React의 단일단계(Thought-Action-Observation)을 표현'''
    def __init__(self,step_num, thought, action,action_input,observation):
        self.step_num = step_num 
        self.thought  =thought
        self.action=action
        self.action_input=action_input
        self.observation=observation
    def display(self):
        print(f'-- step {self.step_num} --')
        print(f'Thought {self.thought}')
        print(f'Action {self.action}[{self.action_input}]')
        print(f'Observation {self.observation}')
        print()
class ReActTrace:
    """ReAct의 전체 추론 과정(trace)을 관리하는 클래스"""
    def __init__(self, question):
        self.question = question
        self.steps = []
        self.final_answer = None

    def add_step(self, thought, action, action_input, observation):
        step = ReActStep(len(self.steps) + 1, thought, action, action_input, observation)
        self.steps.append(step)
        return self

    def finish(self, answer):
        self.final_answer = answer
        return self

    def display(self):
        print("=" * 60)
        print(f"Question: {self.question}")
        print("=" * 60)
        for step in self.steps:
            step.display()
        if self.final_answer:
            print(f"Final Answer: {self.final_answer}")
        print("=" * 60)


# 사용 예시: 다단계 질문에 대한 ReAct 트레이스
trace = ReActTrace("2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?")

trace.add_step(
    thought="이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.",
    action="Search",
    action_input="2024년 노벨 물리학상 수상자",
    observation="2024년 노벨 물리학상은 존 홉필드(John Hopfield)와 제프리 힌턴(Geoffrey Hinton)이 수상했습니다."
)

trace.add_step(
    thought="수상자를 확인했습니다. 이제 이들의 주요 연구 분야를 조회해야 합니다.",
    action="Search",
    action_input="존 홉필드 제프리 힌턴 연구 분야",
    observation="존 홉필드는 홉필드 네트워크(연상 기억 모델)를, 제프리 힌턴은 볼츠만 머신과 역전파 알고리즘 등 인공 신경망 분야를 개척했습니다."
)

trace.finish(
    "2024년 노벨 물리학상 수상자는 존 홉필드와 제프리 힌턴이며, "
    "이들의 주요 연구 분야는 인공 신경망과 기계 학습의 기초 이론입니다."
)

trace.display()        

Question: 2024년 노벨 물리학상 수상자는 누구이며, 그의 주요 연구 분야는?
-- step 1 --
Thought 이 질문에 답하려면 2024년 노벨 물리학상 수상자를 먼저 찾아야 합니다.
Action Search[2024년 노벨 물리학상 수상자]
Observation 2024년 노벨 물리학상은 존 홉필드(John Hopfield)와 제프리 힌턴(Geoffrey Hinton)이 수상했습니다.

-- step 2 --
Thought 수상자를 확인했습니다. 이제 이들의 주요 연구 분야를 조회해야 합니다.
Action Search[존 홉필드 제프리 힌턴 연구 분야]
Observation 존 홉필드는 홉필드 네트워크(연상 기억 모델)를, 제프리 힌턴은 볼츠만 머신과 역전파 알고리즘 등 인공 신경망 분야를 개척했습니다.

Final Answer: 2024년 노벨 물리학상 수상자는 존 홉필드와 제프리 힌턴이며, 이들의 주요 연구 분야는 인공 신경망과 기계 학습의 기초 이론입니다.


In [3]:
# ReAct vs CoT vs 기타 기법 비교 분석표
# 각 기법의 핵심 특성을 체계적으로 비교합니다.

comparison = {
    "기법": ["Zero-shot", "Few-shot", "CoT", "ReAct"],
    "외부 도구 사용": ["X", "X", "X", "O"],
    "추론 과정 명시": ["X", "X", "O", "O"],
    "다단계 정보 수집": ["X", "X", "X", "O"],
    "사실 검증 가능": ["X", "X", "X", "O"],
    "환각 감소": ["낮음", "중간", "중간", "높음"],
    "복잡한 질문 처리": ["제한적", "제한적", "양호", "우수"]
}

# 포맷팅된 비교표 출력
headers = list(comparison.keys())
print(f"{'기법':<12} {'외부 도구 사용':<16} {'추론 과정 명시':<16} {'다단계 정보 수집':<18} {'사실 검증 가능':<16} {'환각 감소':<12} {'복잡한 질문 처리':<16}")
print("-" * 140)
for i in range(len(comparison["기법"])):
    row = [comparison[key][i] for key in headers]
    print(f"{row[0]:<20} {row[1]:<20} {row[2]:<20} {row[3]:<20} {row[4]:<20} {row[5]:<16} {row[6]}")

기법           외부 도구 사용         추론 과정 명시         다단계 정보 수집          사실 검증 가능         환각 감소        복잡한 질문 처리       
--------------------------------------------------------------------------------------------------------------------------------------------
Zero-shot            X                    X                    X                    X                    낮음               제한적
Few-shot             X                    X                    X                    X                    중간               제한적
CoT                  X                    O                    X                    X                    중간               양호
ReAct                O                    O                    O                    O                    높음               우수


# [2교시]

### 1. ReAct 프롬프트 템플릿 설계 원칙
- **시스템 프롬프트의 역할**: 에이전트의 페르소나, 목표, 사용 가능한 도구, 응답 형식 등의 규칙을 정의합니다.
- **도구(Tool) 설명**: 각 도구의 이름, 기능, 입력 형식에 대한 명확한 지시가 포함되어야 합니다.
- **Few-shot 예제**: 모델이 형식에 맞게 출력하도록 가이드 역할을 하는 예제(Thought, Action, Observation 과정 포함)를 제공하는 것이 매우 중요합니다.
- **출력 형식 제약 조건 명시**: Thought, Action, Observation의 정확한 텍스트 패턴과 최종 답변(Finish) 형식을 명시해야 파서가 쉽게 처리할 수 있습니다.

In [4]:
class ReactPromptTemplate:
    def __init__(self):
        self.tools = []
        self.examples = []
        self.system_instruction = ""
    def add_tool(self, name,description,usage_example):
        self.tools.append({'name':name, 'description':description,'usage':usage_example})
        return self
    def add_example(self, question,react_trace):
        self.examples.append({'question':question, 'react_trace':react_trace})
        return self
    def set_system_instruction(self,instruction):
        self.system_instruction = instruction
        return self
    def build(self,user_question):
        prompt_parts = []
        
        # System instruction
        prompt_parts.append(self.system_instruction)
        prompt_parts.append("")
        
        # Tool descriptions
        prompt_parts.append("사용 가능한 도구:")
        for tool in self.tools:            
            prompt_parts.append(f"  - {tool['name']}: {tool['description']}")
            prompt_parts.append(f"    사용법: {tool['usage']}")
        prompt_parts.append("")
        
        # Output format
        prompt_parts.append("출력 형식:")
        prompt_parts.append("Thought: [현재 상황에 대한 추론]")
        prompt_parts.append("Action: [도구 이름][입력값]")
        prompt_parts.append("Observation: [도구 실행 결과]")
        prompt_parts.append("... (필요한 만큼 반복)")
        prompt_parts.append("Thought: [최종 추론]")
        prompt_parts.append("Action: Finish[최종 답변]")
        prompt_parts.append("")
        
        # Few-shot examples
        if self.examples:
            prompt_parts.append("예시:")
            for i, ex in enumerate(self.examples, 1):
                prompt_parts.append(f"--- 예시 {i} ---")
                prompt_parts.append(f"Question: {ex['question']}")
                prompt_parts.append(ex['react_trace'])
                prompt_parts.append("")
        
        # User question
        prompt_parts.append(f"Question: {user_question}")
        
        return "\n".join(prompt_parts)

In [5]:
template = ReactPromptTemplate()
template.set_system_instruction(
    "당신은 주어진 질문에 정학하게 답변하기위해 도구를 사용하는 AI 에이전트입니다\n"
    "반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다"
)
template.add_tool('Search','위키피디아에서 정보를 검색합니다.','Search[검색어]')
template.add_tool('Lookup','현재 문서에서 특정 키워드를 찾습니다.','Lookup[키워드]')
template.add_tool('Calulator','수학 계산을 수행합니다.','Calculate[수식]')

example_trace = """Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이며, 인구는 약 210만 명입니다.]"""

template.add_example("프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?", example_trace)

prompt = template.build("독일의 수도는 어디이며, 면적은 얼마인가?")
print(prompt)


당신은 주어진 질문에 정학하게 답변하기위해 도구를 사용하는 AI 에이전트입니다
반드시 Thought->Action->Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다

사용 가능한 도구:
  - Search: 위키피디아에서 정보를 검색합니다.
    사용법: Search[검색어]
  - Lookup: 현재 문서에서 특정 키워드를 찾습니다.
    사용법: Lookup[키워드]
  - Calulator: 수학 계산을 수행합니다.
    사용법: Calculate[수식]

출력 형식:
Thought: [현재 상황에 대한 추론]
Action: [도구 이름][입력값]
Observation: [도구 실행 결과]
... (필요한 만큼 반복)
Thought: [최종 추론]
Action: Finish[최종 답변]

예시:
--- 예시 1 ---
Question: 프랑스의 수도는 어디이며, 그 도시의 인구는 얼마인가?
Thought: 프랑스의 수도를 먼저 검색해야 합니다.
Action: Search[프랑스 수도]
Observation: 프랑스의 수도는 파리(Paris)입니다.
Thought: 파리의 인구를 추가로 검색해야 합니다.
Action: Search[파리 인구]
Observation: 파리의 인구는 약 210만 명입니다(2023년 기준).
Thought: 필요한 정보를 모두 확보했습니다.
Action: Finish[프랑스의 수도는 파리이며, 인구는 약 210만 명입니다.]

Question: 독일의 수도는 어디이며, 면적은 얼마인가?


### ReAct 출력 파싱
- 정규표현식 기반 : 문자열 패턴 매칭을 통해서 응답에서 Thought, Action, Objservation등의요소를 안정적으로 추출
- 에지케이스 처리 : 특수문자나 잘못된 형식등.. 모델의 예측불가능한 출력을 처리하는 로직이 필요

In [6]:
import re 
from typing import List, Dict ,Optional,Tuple
class ReActParser:
    THOUGHT_PATTERN = re.compile(r'Thought\s*:\s*(.+?)(?=\nAction|$)', re.DOTALL)  
    ACTION_PATTERN = re.compile(r'Action\s*:\s*(\w+)\[(.+?)\]', re.DOTALL)
    OBSERVATION_PATTERN = re.compile(r'Observation\s*:\s*(.+?)(?=\nThought|$)', re.DOTALL)
    FINISH_PATTERN = re.compile(r'Action\s*:\s*Finish\[(.+?)\]', re.DOTALL)
    @classmethod
    def parse_single_step(cls, text: str) -> Dict:
        result = {"thought": None, "action": None, "action_input": None, "observation": None}
        
        thought_match = cls.THOUGHT_PATTERN.search(text)
        if thought_match:
            result["thought"] = thought_match.group(1).strip()
        
        action_match = cls.ACTION_PATTERN.search(text)
        if action_match:
            result["action"] = action_match.group(1).strip()
            result["action_input"] = action_match.group(2).strip()
        
        obs_match = cls.OBSERVATION_PATTERN.search(text)
        if obs_match:
            result["observation"] = obs_match.group(1).strip()
        
        return result
    
    @classmethod
    def parse_full_trace(cls, text: str) -> Dict:
        steps = []
        final_answer = None
        
        # Check for finish action
        finish_match = cls.FINISH_PATTERN.search(text)
        if finish_match:
            final_answer = finish_match.group(1).strip()
        
        # Split by Thought markers
        thought_splits = re.split(r'(?=Thought\s*:)', text)
        thought_splits = [s.strip() for s in thought_splits if s.strip()]
        
        for split in thought_splits:
            step = cls.parse_single_step(split)
            if step["thought"]:  # Valid step
                steps.append(step)
        
        return {
            "steps": steps,
            "final_answer": final_answer,
            "num_steps": len(steps)
        }
    
    @classmethod
    def is_finished(cls, text: str) -> bool:
        return bool(cls.FINISH_PATTERN.search(text))
    
    @classmethod
    def extract_action(cls, text: str) -> Optional[Tuple[str, str]]:
        match = cls.ACTION_PATTERN.search(text)
        if match:
            return (match.group(1).strip(), match.group(2).strip())
        return None

In [7]:
test_trace = """Thought: 질문에 답하려면 양자 컴퓨팅의 기본 원리를 먼저 조사해야 합니다.
Action: Search[양자 컴퓨팅 기본 원리]
Observation: 양자 컴퓨팅은 큐비트(qubit)를 사용하며, 중첩(superposition)과 얽힘(entanglement)의 원리를 활용합니다.
Thought: 기본 원리를 확인했습니다. 이제 기존 컴퓨팅과의 차이점을 조사해야 합니다.
Action: Search[양자 컴퓨팅 기존 컴퓨팅 차이]
Observation: 기존 컴퓨팅은 비트(0 또는 1)를 사용하지만, 양자 컴퓨팅은 큐비트를 사용해 동시에 여러 상태를 표현할 수 있어 특정 문제에서 기하급수적 속도 향상이 가능합니다.
Thought: 충분한 정보를 확보했으므로 최종 답변을 작성합니다.
Action: Finish[양자 컴퓨팅은 큐비트를 기반으로 중첩과 얽힘을 활용하며, 기존 비트 기반 컴퓨팅보다 특정 문제에서 기하급수적으로 빠른 연산이 가능합니다.]"""

result = ReActParser.parse_full_trace(test_trace)
print("=== 파싱 결과 ===")
print(f"총 단계 수: {result['num_steps']}")
for i, step in enumerate(result['steps'], 1):
    print(f"\n--- Step {i} ---")
    print(f"  Thought   : {step['thought']}")
    print(f"  Action    : {step['action']}")
    print(f"  Input     : {step['action_input']}")
    print(f"  Observation: {step['observation']}")
print(f"\n최종 답변: {result['final_answer']}")
print(f"종료 여부: {ReActParser.is_finished(test_trace)}")

=== 파싱 결과 ===
총 단계 수: 3

--- Step 1 ---
  Thought   : 질문에 답하려면 양자 컴퓨팅의 기본 원리를 먼저 조사해야 합니다.
  Action    : Search
  Input     : 양자 컴퓨팅 기본 원리
  Observation: 양자 컴퓨팅은 큐비트(qubit)를 사용하며, 중첩(superposition)과 얽힘(entanglement)의 원리를 활용합니다.

--- Step 2 ---
  Thought   : 기본 원리를 확인했습니다. 이제 기존 컴퓨팅과의 차이점을 조사해야 합니다.
  Action    : Search
  Input     : 양자 컴퓨팅 기존 컴퓨팅 차이
  Observation: 기존 컴퓨팅은 비트(0 또는 1)를 사용하지만, 양자 컴퓨팅은 큐비트를 사용해 동시에 여러 상태를 표현할 수 있어 특정 문제에서 기하급수적 속도 향상이 가능합니다.

--- Step 3 ---
  Thought   : 충분한 정보를 확보했으므로 최종 답변을 작성합니다.
  Action    : Finish
  Input     : 양자 컴퓨팅은 큐비트를 기반으로 중첩과 얽힘을 활용하며, 기존 비트 기반 컴퓨팅보다 특정 문제에서 기하급수적으로 빠른 연산이 가능합니다.
  Observation: None

최종 답변: 양자 컴퓨팅은 큐비트를 기반으로 중첩과 얽힘을 활용하며, 기존 비트 기반 컴퓨팅보다 특정 문제에서 기하급수적으로 빠른 연산이 가능합니다.
종료 여부: True


### 출력 결과 분석
파서는 LLM이 출력한 비정형 텍스트에서 각 ReAct 루프의 구성요소를 객체 형태로 성공적으로 추출했습니다. 이는 파이썬 코드 레벨에서 Tool 객체에 `action_input`을 전달하고 실행 결과를 다시 모델에 반환하기 위해 필수적인 과정입니다.

In [8]:
# Edge case tests
print("=== 에지 케이스 테스트 ===")

# Case 1: Thought only (action not yet generated)
case1 = "Thought: 먼저 검색을 해야 합니다."
result1 = ReActParser.parse_single_step(case1)
print(f"\n[Case 1] Thought만 있는 경우:")
print(f"  Thought: {result1['thought']}")
print(f"  Action: {result1['action']}")

# Case 2: Direct finish
case2 = "Thought: 이미 알고 있는 정보입니다.\nAction: Finish[지구의 위성은 달입니다.]"
result2 = ReActParser.parse_single_step(case2)
print(f"\n[Case 2] 바로 종료하는 경우:")
print(f"  Thought: {result2['thought']}")
print(f"  Action: {result2['action']}")
print(f"  Input: {result2['action_input']}")
print(f"  Finished: {ReActParser.is_finished(case2)}")

# Case 3: Action extraction
case3 = "Thought: 계산이 필요합니다.\nAction: Calculate[15 * 24 + 30]"
action_result = ReActParser.extract_action(case3)
print(f"\n[Case 3] Action 추출:")
print(f"  Tool: {action_result[0]}, Input: {action_result[1]}")

# Case 4: Invalid format
case4 = "이것은 ReAct 형식이 아닌 일반 텍스트입니다."
result4 = ReActParser.parse_single_step(case4)
print(f"\n[Case 4] 잘못된 형식:")
print(f"  Thought: {result4['thought']}")
print(f"  Action: {result4['action']}")
print(f"  모든 필드가 None: {all(v is None for v in result4.values())}")

=== 에지 케이스 테스트 ===

[Case 1] Thought만 있는 경우:
  Thought: 먼저 검색을 해야 합니다.
  Action: None

[Case 2] 바로 종료하는 경우:
  Thought: 이미 알고 있는 정보입니다.
  Action: Finish
  Input: 지구의 위성은 달입니다.
  Finished: True

[Case 3] Action 추출:
  Tool: Calculate, Input: 15 * 24 + 30

[Case 4] 잘못된 형식:
  Thought: None
  Action: None
  모든 필드가 None: True


### 프롬프트 변형과 최적화
프롬프트의 디테일(도구 설명의 구체성, 시스템 프롬프트의 지시사항, 예제 등)에 따라 에이전트의 성능이 달라집니다.

In [9]:
# Variation 1: Minimal prompt
minimal_template = ReactPromptTemplate()
minimal_template.set_system_instruction("도구를 사용하여 질문에 답하세요.")
minimal_template.add_tool("Search", "검색", "Search[query]")

minimal_prompt = minimal_template.build("한국의 GDP는 얼마인가?")

# Variation 2: Standard prompt
standard_template = ReactPromptTemplate()
standard_template.set_system_instruction(
    "당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다.\n"
    "반드시 Thought -> Action -> Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다."
)
standard_template.add_tool("Search", "위키피디아에서 정보를 검색합니다.", "Search[검색어]")
standard_template.add_tool("Calculate", "수학 계산을 수행합니다.", "Calculate[수식]")

standard_example = """Thought: GDP 정보를 검색해야 합니다.
Action: Search[미국 GDP]
Observation: 미국의 GDP는 약 25조 달러입니다(2023년 기준).
Thought: 정보를 확보했습니다.
Action: Finish[미국의 GDP는 약 25조 달러입니다.]"""
standard_template.add_example("미국의 GDP는 얼마인가?", standard_example)

standard_prompt = standard_template.build("한국의 GDP는 얼마인가?")

# Variation 3: Verbose prompt
verbose_template = ReactPromptTemplate()
verbose_template.set_system_instruction(
    "당신은 정확하고 신뢰할 수 있는 정보를 제공하기 위해 외부 도구를 활용하는 AI 에이전트입니다.\n"
    "다음 규칙을 반드시 준수하세요:\n"
    "1. 모든 답변은 도구를 통해 검증된 정보에 기반해야 합니다.\n"
    "2. 추측이나 가정을 하지 마세요. 확인되지 않은 정보는 명시적으로 표시하세요.\n"
    "3. Thought -> Action -> Observation 순서를 엄격히 따르세요.\n"
    "4. 최종 답변은 반드시 Finish 액션으로 제출하세요.\n"
    "5. 각 Thought에서는 현재까지 확보한 정보와 다음에 필요한 정보를 명확히 서술하세요."
)
verbose_template.add_tool(
    "Search",
    "위키피디아 데이터베이스에서 키워드 기반으로 관련 문서를 검색합니다. "
    "검색어는 구체적이고 명확할수록 좋은 결과를 얻을 수 있습니다.",
    "Search[구체적인 검색어]"
)
verbose_template.add_tool(
    "Lookup",
    "현재 조회 중인 문서 내에서 특정 키워드가 포함된 단락을 찾습니다. "
    "Search로 문서를 먼저 조회한 후 사용해야 합니다.",
    "Lookup[문서 내 검색 키워드]"
)
verbose_template.add_tool(
    "Calculate",
    "수학 연산을 수행합니다. 사칙연산, 거듭제곱, 백분율 계산 등을 지원합니다. "
    "수식은 Python 문법을 따릅니다.",
    "Calculate[수학 수식, 예: 100 * 1.05 ** 10]"
)

verbose_example = """Thought: 한국의 GDP에 대한 질문입니다. 정확한 최신 데이터를 찾기 위해 검색이 필요합니다.
Action: Search[대한민국 GDP 2023년]
Observation: 대한민국의 2023년 명목 GDP는 약 1조 7,130억 달러로, 세계 13위 수준입니다.
Thought: GDP 총액을 확인했습니다. 더 정확한 답변을 위해 1인당 GDP도 확인해 보겠습니다.
Action: Search[대한민국 1인당 GDP 2023년]
Observation: 대한민국의 2023년 1인당 명목 GDP는 약 33,147 달러입니다.
Thought: 총 GDP와 1인당 GDP 정보를 모두 확보했습니다. 종합적인 답변을 작성할 수 있습니다.
Action: Finish[대한민국의 2023년 명목 GDP는 약 1조 7,130억 달러(세계 13위)이며, 1인당 GDP는 약 33,147 달러입니다.]"""
verbose_template.add_example("한국의 GDP는 얼마인가?", verbose_example)

verbose_prompt = verbose_template.build("한국의 GDP는 얼마인가?")

# Compare all three
prompts = {
    "Minimal (최소형)": minimal_prompt,
    "Standard (표준형)": standard_prompt,
    "Verbose (상세형)": verbose_prompt
}

for name, p in prompts.items():
    line_count = len(p.split('\n'))
    char_count = len(p)
    print(f"\n{'=' * 60}")
    print(f"[{name}] - {line_count}줄, {char_count}자")
    print(f"{'=' * 60}")
    print(p)


[Minimal (최소형)] - 15줄, 226자
도구를 사용하여 질문에 답하세요.

사용 가능한 도구:
  - Search: 검색
    사용법: Search[query]

출력 형식:
Thought: [현재 상황에 대한 추론]
Action: [도구 이름][입력값]
Observation: [도구 실행 결과]
... (필요한 만큼 반복)
Thought: [최종 추론]
Action: Finish[최종 답변]

Question: 한국의 GDP는 얼마인가?

[Standard (표준형)] - 27줄, 588자
당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다.
반드시 Thought -> Action -> Observation 순서를 따르며, 최종 답변은 Finish 액션으로 제출합니다.

사용 가능한 도구:
  - Search: 위키피디아에서 정보를 검색합니다.
    사용법: Search[검색어]
  - Calculate: 수학 계산을 수행합니다.
    사용법: Calculate[수식]

출력 형식:
Thought: [현재 상황에 대한 추론]
Action: [도구 이름][입력값]
Observation: [도구 실행 결과]
... (필요한 만큼 반복)
Thought: [최종 추론]
Action: Finish[최종 답변]

예시:
--- 예시 1 ---
Question: 미국의 GDP는 얼마인가?
Thought: GDP 정보를 검색해야 합니다.
Action: Search[미국 GDP]
Observation: 미국의 GDP는 약 25조 달러입니다(2023년 기준).
Thought: 정보를 확보했습니다.
Action: Finish[미국의 GDP는 약 25조 달러입니다.]

Question: 한국의 GDP는 얼마인가?

[Verbose (상세형)] - 37줄, 1291자
당신은 정확하고 신뢰할 수 있는 정보를 제공하기 위해 외부 도구를 활용하는 AI 에이전트입니다.
다음 규칙을 반드시 준수하세요:
1. 모든 답변은 도구를 통해 검증된 정보

# [3교시]

## 1. Tool(도구)의 정의와 역할

### 1.1 ReAct에서 Tool이란 무엇인가

ReAct 프레임워크에서 **Tool(도구)** 은 LLM이 외부 세계와 상호작용하기 위해 호출하는 실행 가능한 함수 또는 서비스를 의미한다. LLM은 자연어를 이해하고 생성하는 능력이 뛰어나지만, 다음과 같은 한계를 가진다.

- **수학 연산**: 복잡한 계산에서 오류를 범할 수 있다
- **최신 정보**: 학습 데이터 이후의 정보를 알 수 없다
- **외부 시스템 접근**: 데이터베이스, 파일 시스템, 웹 검색 등에 직접 접근할 수 없다
- **정확한 데이터 조회**: 정확한 수치나 사실 정보를 보장할 수 없다

Tool은 이러한 한계를 보완하여, LLM의 추론 능력과 외부 도구의 실행 능력을 결합하는 핵심 요소이다.

### 1.2 Tool의 필수 요소

모든 Tool은 다음 5가지 요소를 갖추어야 한다.

| 요소 | 설명 | 예시 |
|------|------|------|
| **이름(Name)** | 도구를 식별하는 고유한 이름 | `Calculator`, `WebSearch` |
| **설명(Description)** | 도구의 기능을 LLM이 이해할 수 있도록 기술 | "수학 수식을 계산합니다" |
| **입력 스키마(Input Schema)** | 도구가 받는 입력의 형식과 제약 조건 | `{"type": "string"}` |
| **실행 로직(Execute Logic)** | 입력을 받아 실제 작업을 수행하는 코드 | `eval(expression)` |
| **출력 형식(Output Format)** | 결과를 문자열로 반환하는 규칙 | `"계산 결과: 42"` |

### 1.3 Tool이 LLM의 한계를 보완하는 방식

ReAct에서 LLM과 Tool의 협업은 다음과 같은 흐름을 따른다.

```
LLM(Thought) -> 어떤 도구를 사용할지 결정
LLM(Action)  -> 도구 이름과 입력값을 지정
Tool(실행)    -> 실제 연산/조회 수행
LLM(관찰)    -> 도구 결과를 받아 다음 추론에 활용
```

이 구조에서 LLM은 **"무엇을 해야 하는가"**를 결정하는 두뇌 역할을 하고, Tool은 **"실제로 수행"**하는 손과 발 역할을 한다.

## 2. Tool 스키마 설계

### 2.1 JSON Schema 기반 Tool 정의

도구를 체계적으로 정의하기 위해 JSON Schema 형식을 사용한다. 이는 OpenAI의 Function Calling이나 LangChain 등 주요 프레임워크에서도 채택하고 있는 표준 방식이다.

```json
{
  "name": "Calculator",
  "description": "수학 수식을 계산합니다.",
  "parameters": {
    "type": "string",
    "description": "계산할 수학 수식 (예: 2 + 3 * 4)"
  }
}
```

### 2.2 매개변수 타입과 필수/선택 지정

복잡한 도구의 경우, 매개변수를 세부적으로 정의할 수 있다.

```json
{
  "name": "WebSearch",
  "description": "웹에서 정보를 검색합니다.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {"type": "string", "description": "검색 질의"},
      "max_results": {"type": "integer", "description": "최대 결과 수", "default": 5}
    },
    "required": ["query"]
  }
}
```

### 2.3 Tool 설명문 작성 가이드라인

도구의 설명문은 LLM이 적절한 도구를 선택하는 데 결정적인 역할을 한다. 다음 원칙을 따르는 것이 좋다.

1. **명확성**: 도구가 무엇을 하는지 한 문장으로 설명한다
2. **입력 예시**: 어떤 형식의 입력을 기대하는지 예시를 제공한다
3. **제한 사항**: 도구가 처리할 수 없는 경우를 명시한다
4. **사용 시점**: 언제 이 도구를 사용해야 하는지 안내한다

### 기본 Tool 클래스 설계

이제 파이썬의 추상 클래스(ABC)를 활용하여 Tool의 기본 구조를 설계하고, 세 가지 구체적인 도구를 구현한다.

- **CalculatorTool**: 수학 수식을 평가하는 계산기
- **StringProcessorTool**: 문자열 통계를 분석하는 도구
- **DictionaryTool**: 미리 정의된 사전에서 용어를 검색하는 도구

In [10]:
# abc  (Abstract Base Class)  - 추상클래스
from abc import ABC, abstractmethod
from typing import Any, Dict,Optional
import json
class BaseTool(ABC):  # 추상클래스
    '''React 에이전트에서 사용하는 도구의 기본 클래스'''
    def __init__(self,name:str, description:str):
        self.name = name; self.description = description
    @abstractmethod
    def execute(self, input_text:str) ->str:
        '''도구를 실행하고 결과를 문자열로 반환'''
        pass

    def get_schema(self)->Dict:
        '''도구의 스키마를 반환'''
        return{
            'name' : self.name,
            'description':self.description,
            'parameters':self._get_parameters()
        }
    def _get_parameters(self)->str:
        return {'type':'string','description':'입력테스트'}
    def __repr__(self):  # 객체를 문자열로 표현할때 사용되는 callback함수
        return super().__repr__()

In [11]:
class Person:
    def __init__(self,name):
        self.name = name
p = Person("hong gil dong")        
print(p)

In [12]:
class Person:
    def __init__(self,name):
        self.name = name
    def __repr__(self):
        return f'Person(name={self.name})'
p = Person("hong gil dong")        
print(p)

Person(name=hong gil dong)


In [15]:
class CalculatorTool(BaseTool):
    def __init__(self):
        super().__init__(
            name='Calcualtor',
            description='수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.'
        )
    def execute(self, input_text:str) ->str:
        try:
            # 안전한 수식 평가를 위해 허용된 연산자만 사용
            allowed_chars = set('0123456789+-*/.() ')
            if not all(c in allowed_chars for c in input_text):
                return f"오류: 허용되지 않는 문자가 포함되어 있습니다."
            result = eval(input_text)
            return f"계산 결과: {result}"
        except Exception as e:
            return f"계산 오류: {str(e)}"
class StringProcessorTool(BaseTool):
    """문자열 처리 도구"""
    
    def __init__(self):
        super().__init__(
            name="StringProcessor",
            description="문자열의 길이, 단어 수, 글자 수 등을 분석합니다."
        )
    
    def execute(self, input_text: str) -> str:
        char_count = len(input_text)
        word_count = len(input_text.split())
        line_count = len(input_text.splitlines())
        return (f"문자 수: {char_count}, 단어 수: {word_count}, "
                f"줄 수: {line_count}")

class DictionaryTool(BaseTool):
    """미리 정의된 사전에서 정보를 검색하는 도구"""
    
    def __init__(self):
        super().__init__(
            name="Dictionary",
            description="내장 사전에서 용어의 정의를 검색합니다."
        )
        self.entries = {
            "인공지능": "인간의 학습, 추론, 지각 능력을 컴퓨터로 구현하는 기술",
            "머신러닝": "데이터로부터 패턴을 학습하여 예측이나 결정을 수행하는 AI의 하위 분야",
            "딥러닝": "다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야",
            "자연어처리": "컴퓨터가 인간의 언어를 이해하고 생성하는 AI 기술",
            "강화학습": "환경과의 상호작용을 통해 보상을 최대화하는 행동을 학습하는 방법",
            "트랜스포머": "어텐션 메커니즘 기반의 신경망 아키텍처로, 현대 NLP의 기반",
            "LLM": "대규모 텍스트 데이터로 사전 학습된 거대 언어 모델",
            "프롬프트": "LLM에 입력되는 지시문 또는 질의문",
            "ReAct": "추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법"
        }
    
    def execute(self, input_text: str) -> str:
        term = input_text.strip()
        if term in self.entries:
            return f"{term}: {self.entries[term]}"
        # Partial match
        matches = [k for k in self.entries if term in k or k in term]
        if matches:
            results = [f"{k}: {self.entries[k]}" for k in matches]
            return "관련 항목:\n" + "\n".join(results)
        return f"'{term}'에 대한 정보를 찾을 수 없습니다."        

In [16]:
# Test Tool
calc = CalculatorTool()
print(f'schema : {json.dumps(calc.get_schema(),ensure_ascii=False,indent=3 )}')
print(f'실행 : {calc.execute("(15+25)*3")}')
print(f'오류 : {calc.execute("import os")}')

string_proc = StringProcessorTool()
print(f'실행:{string_proc.execute("ReAct는 대세입니다.")}')

dict_tool = DictionaryTool()
print(f'실행:{dict_tool.execute("ReAct는 대세입니다. 트랜스포머")}')

schema : {
   "name": "Calcualtor",
   "description": "수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.",
   "parameters": {
      "type": "string",
      "description": "입력테스트"
   }
}
실행 : 계산 결과: 120
오류 : 오류: 허용되지 않는 문자가 포함되어 있습니다.
실행:문자 수: 13, 단어 수: 2, 줄 수: 1
실행:관련 항목:
트랜스포머: 어텐션 메커니즘 기반의 신경망 아키텍처로, 현대 NLP의 기반
ReAct: 추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법


# [4교시]

## 3. Tool Registry 패턴

### 3.1 Registry의 개념과 필요성

실제 ReAct 에이전트는 여러 도구를 동시에 관리해야 한다. 이때 **Registry(레지스트리)** 패턴을 사용하면 다음과 같은 이점이 있다.

1. **중앙 집중 관리**: 모든 도구를 한 곳에서 등록하고 조회할 수 있다
2. **동적 확장**: 런타임에 새로운 도구를 추가하거나 제거할 수 있다
3. **일관된 인터페이스**: 도구 이름만으로 실행할 수 있는 통일된 API를 제공한다
4. **오류 처리**: 존재하지 않는 도구 호출 등의 예외를 중앙에서 처리한다

### 3.2 도구 등록/조회/실행 흐름

```
1. register(tool)  -> 도구를 이름으로 등록
2. get(name)       -> 이름으로 도구 객체 조회
3. execute(name, input) -> 이름과 입력으로 도구 실행
4. list_tools()    -> 등록된 모든 도구 목록 반환
```

이 흐름은 ReAct 루프에서 LLM이 `Action: ToolName[input]` 형식으로 도구를 호출할 때 그대로 적용된다.

In [17]:
class ToolRegistry:
    '''도구를 등록하고 관리하는 레지스트리'''
    def __init__(self):
        self._tools:Dict[str,BaseTool] = {}
    def register(self, tool:BaseTool) -> 'ToolRegistry':
        self._tools[tool.name] = tool        
        return self
    def get(self,name:str) -> Optional[BaseTool]:
        return self._tools.get(name)
    def execute(self, tool_name:str, input_text:str)->str:
        tool = self.get(tool_name)
        if tool is None:
            return f'오류 : {tool_name} 도구를 찾을수 없습니다. 사용가능한 도구 : {self._tools.keys()}'
        try:
            return tool.execute(input_text)
        except Exception as e:
            return f'실행 오류 : {e}'
    def list_tools(self)->str:
        descriptions = []
        for name, tool in self._tools.items():
            descriptions.append(f'  - {name} : {tool.description}')
        return "사용가능한도구\n" + "\n".join(descriptions)
    def get_all_schemas(self)->list:
        return [ tool.get_schema() for tool in self._tools.values()]
    
# Create and populate registry
print("=== Tool Registry 구성 ===")
registry = ToolRegistry()
registry.register(CalculatorTool())
registry.register(StringProcessorTool())
registry.register(DictionaryTool())

print(f"\n{registry.list_tools()}")

print("\n=== Registry를 통한 도구 실행 ===")
print(registry.execute("Calculator", "2 ** 10"))
print(registry.execute("Dictionary", "딥러닝"))
print(registry.execute("UnknownTool", "test"))    

=== Tool Registry 구성 ===

사용가능한도구
  - Calcualtor : 수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.
  - StringProcessor : 문자열의 길이, 단어 수, 글자 수 등을 분석합니다.
  - Dictionary : 내장 사전에서 용어의 정의를 검색합니다.

=== Registry를 통한 도구 실행 ===
오류 : Calculator 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])
딥러닝: 다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야
오류 : UnknownTool 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])


## 4. Tool 체이닝

### 4.1 체이닝의 개념

**Tool 체이닝(Tool Chaining)** 이란 하나의 도구 실행 결과를 다른 도구의 입력으로 사용하는 패턴을 말한다. 복잡한 문제를 해결할 때, 단일 도구만으로는 충분하지 않은 경우가 많다.

예를 들어 다음과 같은 질문을 생각해보자.

> "한국 GDP를 인구수로 나누면 1인당 GDP는 얼마인가?"

이 질문에 답하려면:
1. 지식베이스에서 한국 GDP 조회 (KnowledgeBase 도구)
2. 지식베이스에서 한국 인구수 조회 (KnowledgeBase 도구)
3. GDP / 인구수 계산 (Calculator 도구)

이처럼 여러 도구를 순차적으로 호출하는 것이 체이닝이다.

### 4.2 ReAct에서의 체이닝

ReAct 프레임워크에서는 LLM이 자연스럽게 체이닝을 수행한다. 매 단계마다 이전 Observation을 참고하여 다음 Action을 결정하기 때문에, 개발자가 체이닝 로직을 명시적으로 구현할 필요가 없다. LLM 자체가 체이닝의 "오케스트레이터" 역할을 한다.

In [18]:
def tool_chain(registry, steps):
    """여러 도구를 순차적으로 실행하는 체이닝 함수
    
    Args:
        registry: ToolRegistry 인스턴스
        steps: (tool_name, input_data) 리스트
               input_data가 callable이면 이전 결과를 인자로 받음
    """
    print("=== Tool 체이닝 실행 ===")
    previous_result = None
    
    for i, (tool_name, input_data) in enumerate(steps, 1):
        # input_data가 callable이면 이전 결과를 사용하여 입력 생성
        if callable(input_data):
            actual_input = input_data(previous_result)
        else:
            actual_input = input_data
        
        print(f"\n[Chain Step {i}] {tool_name}")
        print(f"  입력: {actual_input}")
        result = registry.execute(tool_name, actual_input)
        print(f"  출력: {result}")
        previous_result = result
    
    print(f"\n최종 결과: {previous_result}")
    return previous_result

# 체이닝 예시 1: 계산 결과 -> 문자열 분석
print("[예시 1] 계산 결과를 문자열로 분석")
chain_steps = [
    ("Calculator", "(100 + 200 + 300) / 3"),
    ("StringProcessor", lambda prev: prev),  # 이전 결과를 문자열로 분석
]
tool_chain(registry, chain_steps)

# 체이닝 예시 2: 사전 검색 -> 결과 분석 -> 계산
print("\n" + "-" * 40)
print("\n[예시 2] 사전 검색 -> 결과 분석 -> 계산")
chain_steps_2 = [
    ("Dictionary", "LLM"),
    ("StringProcessor", lambda prev: prev),
    ("Calculator", "7 * 8 + 6"),
]
tool_chain(registry, chain_steps_2)

[예시 1] 계산 결과를 문자열로 분석
=== Tool 체이닝 실행 ===

[Chain Step 1] Calculator
  입력: (100 + 200 + 300) / 3
  출력: 오류 : Calculator 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])

[Chain Step 2] StringProcessor
  입력: 오류 : Calculator 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])
  출력: 문자 수: 99, 단어 수: 12, 줄 수: 1

최종 결과: 문자 수: 99, 단어 수: 12, 줄 수: 1

----------------------------------------

[예시 2] 사전 검색 -> 결과 분석 -> 계산
=== Tool 체이닝 실행 ===

[Chain Step 1] Dictionary
  입력: LLM
  출력: LLM: 대규모 텍스트 데이터로 사전 학습된 거대 언어 모델

[Chain Step 2] StringProcessor
  입력: LLM: 대규모 텍스트 데이터로 사전 학습된 거대 언어 모델
  출력: 문자 수: 33, 단어 수: 9, 줄 수: 1

[Chain Step 3] Calculator
  입력: 7 * 8 + 6
  출력: 오류 : Calculator 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])

최종 결과: 오류 : Calculator 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])


"오류 : Calculator 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])"

## 5. 샘플 데이터 기반 ReAct 시뮬레이션 통합

이제 지금까지 구현한 모든 요소를 하나로 통합하여, Tool Registry를 사용하는 완전한 ReAct 시뮬레이션을 수행한다.

이 시뮬레이션에서는 LLM 없이 **미리 정의된 Thought-Action 시퀀스**를 사용하지만, 도구 실행은 실제 ToolRegistry를 통해 이루어진다. 이를 통해 ReAct 루프의 구조를 명확하게 이해할 수 있다.

시뮬레이션 흐름:
1. 질문이 주어진다
2. 각 단계에서 Thought(추론)가 출력된다
3. Action이 Registry를 통해 실행되고 Observation이 반환된다
4. `Finish` 액션이 나오면 최종 답변을 출력하고 종료한다

In [19]:
import re

def simulate_react_with_tools(question, steps, registry):
    """Tool Registry를 사용한 ReAct 시뮬레이션"""
    print(f"Question: {question}")
    print("=" * 50)
    
    for i, step in enumerate(steps, 1):
        print(f"\n--- Step {i} ---")
        print(f"Thought: {step['thought']}")
        
        if step['action'] == 'Finish':
            print(f"Action: Finish[{step['input']}]")
            print(f"\n{'=' * 50}")
            print(f"Final Answer: {step['input']}")
            return step['input']
        
        print(f"Action: {step['action']}[{step['input']}]")
        observation = registry.execute(step['action'], step['input'])
        print(f"Observation: {observation}")
    
    return None

# 시뮬레이션 실행
steps = [
    {
        "thought": "(100 + 250) * 0.1 의 값을 계산해야 합니다.",
        "action": "Calculator",
        "input": "(100 + 250) * 0.1"
    },
    {
        "thought": "계산 결과를 확인했습니다. 이제 ReAct에 대한 정의를 찾아보겠습니다.",
        "action": "Dictionary",
        "input": "ReAct"
    },
    {
        "thought": "필요한 정보를 모두 확인했습니다.",
        "action": "Finish",
        "input": "(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다."
    }
]

simulate_react_with_tools(
    "(100+250)*0.1의 값은 얼마이며, ReAct의 정의는 무엇인가?",
    steps,
    registry
)

Question: (100+250)*0.1의 값은 얼마이며, ReAct의 정의는 무엇인가?

--- Step 1 ---
Thought: (100 + 250) * 0.1 의 값을 계산해야 합니다.
Action: Calculator[(100 + 250) * 0.1]
Observation: 오류 : Calculator 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calcualtor', 'StringProcessor', 'Dictionary'])

--- Step 2 ---
Thought: 계산 결과를 확인했습니다. 이제 ReAct에 대한 정의를 찾아보겠습니다.
Action: Dictionary[ReAct]
Observation: ReAct: 추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법

--- Step 3 ---
Thought: 필요한 정보를 모두 확인했습니다.
Action: Finish[(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.]

Final Answer: (100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.


'(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.'

# [5교시]

### 1. OpenAI Chat Completions API 구조
- **메시지 역할(Role)**: `system` (기본 지시사항), `user` (사용자 입력), `assistant` (LLM의 응답)
- **주요 파라미터**: `temperature` (응답의 무작위성 제어, ReAct에서는 0에 가깝게 설정하는 것이 유리), `max_tokens` 등
- **API 호출 구조**: `openai` 파이썬 패키지를 이용한 기본 호출 방법 소개

In [23]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client  = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
# test
response = client.chat.completions.create(
    model='gpt-5.4-nano',
    messages=[{'role':'user','content':'오늘의 날씨정보를 검색해서 알려주세요'}],
    temperature=0,    
)
print(response.choices[0].message.content)
print(f'토큰사용량 : {response.usage}')

죄송하지만, 저는 현재 **실시간 웹 검색으로 오늘의 날씨를 직접 조회**할 수는 없어요. 대신 아래 정보를 주시면 **바로 확인할 수 있는 방법**(또는 지역 기준으로 정리된 안내)을 도와드릴게요.

1) **어느 지역** 날씨가 필요하세요? (예: 서울 강남구, 부산 해운대구, 대전 전체 등)  
2) **원하는 시간대**가 있나요? (지금 / 오늘 / 시간대별 / 강수확률 포함 등)

원하시면, 아래 중 하나로 진행할 수 있어요:
- **지역을 알려주시면**: 제가 그 지역의 날씨를 확인할 수 있는 **가장 빠른 조회 경로(앱/웹/검색어 형태)**를 정확히 안내
- **사용 중인 서비스**가 있으면: (예: 네이버 날씨, 기상청, 카카오맵 등) 그 서비스 기준으로 **어디를 보면 되는지** 안내

지역을 알려주실래요?
토큰사용량 : CompletionUsage(completion_tokens=231, prompt_tokens=16, total_tokens=247, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


In [24]:
response.usage.total_tokens

247

### 2. ReAct 시스템 프롬프트 설계
시스템 프롬프트에는 다음이 반드시 포함되어야 합니다:
- 에이전트의 역할 지정
- 사용 가능한 도구(Tool) 목록과 그 사용법
- 반드시 따라야 하는 출력 형식 (Thought -> Action -> Observation 패턴)

### 3. ReAct 루프 직접 구현
- **while 또는 for 루프 기반 에이전트 구조**: 에이전트가 `Finish` 액션을 낼 때까지 계속해서 응답을 생성하고 도구를 실행하는 과정을 반복합니다.
- **종료 조건**: 모델의 출력에서 `Action: Finish`가 감지되면 무한 루프를 종료하고 최종 답을 반환합니다.
- **최대 반복 횟수 제한**: 무한 루프를 방지하기 위해 최대 스텝 수를 지정합니다.

In [45]:
import re
class SimpleReactAgent:    
    def __init__(self,model = 'gpt-5.4-nano', max_steps=5):        
        self.model = model
        self.max_steps = max_steps
        self.tools = {}
        self.systemp_prompt = ''
    def register_tool(self, name,func,description):
        self.tools[name] = {'func':func, 'description':description}
    def _build_system_prompt(self):
        tool_desc = '\n'.join([ f"- {name}:{info['description']}" for name, info in self.tools.items()  ])
        self.systemp_prompt = f'''당신은 주어진 질문에 정확하게 답변하기 위해 도구를 사용하는 AI 에이전트입니다.
        사용 가능한 도구:
        {tool_desc}

        반드시 다음 형식을 따라 응답하세요:
        Thought: [현재 상황에 대한 추론]
        Action: [도구이름][입력값]

        도구 실행결과는 Observation으로 제공됩니다
        최종 답변을 제출할 때는 반드시 다음 형식을 사용하세요:
        Thought: [최종 추론]
        Action: Finish[최종 답변]

        중요: 한 번에 하나의 Action만 수행하세요
        '''
    def _parse_action(self, text):
        match = re.search(r'Action\s*:\s*(\w+)\[(.+?)\]', text, re.DOTALL)
        if match:
            return match.group(1).strip(), match.group(2).strip()
        return None, None
    def _execute_tool(self, tool_name, tool_input):
        if tool_name == 'Finish':
            return None
        elif tool_name in self.tools:
            try:
                return self.tools[tool_name]['func'](tool_input)
            except Exception as e:
                return f'도구 실행 오류:{e}'
    def run(self, question):
        self._build_system_prompt()
        messages = [
            {'role':'system','content':self.systemp_prompt},
            {'role':'user','content':f'Question:{question}'}
        ]
        print(f'Question:{question}')
        print('='*60)
        for step in range(self.max_steps):
            response = client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0
            )            
            assistant_msg = response.choices[0].message.content
            messages.append({'role':'assistant','content':assistant_msg})

            print(f'\n -- Step {step+1} --')
            print(assistant_msg)

            action_name, action_input =  self._parse_action(assistant_msg)

            if action_name =='Finish':
                print(f"\n{'='*60}")
                print(f'final Answer : {action_input}')                
                return action_input
            elif action_name:
                observation = self._execute_tool(action_name,action_input)
                obs_msg = f'Observation: {observation}'
                messages.append({'role':'user','content':obs_msg})
                print(f'\n{obs_msg}')
            else:
                messages.append({'role':'user', 'content':'형식오류:Action:도구이름[입력값] 형식으로 응답해주세요'})
        print('\n최대 단계수에 도달했습니다.')
        return None

# [6교시]

In [46]:
# Define simple tools
def calculator(expression):
    allowed = set('0123456789+-*/.() ')
    if all(c in allowed for c in expression):
        return str(eval(expression))
    return "허용되지 않는 수식입니다."

def knowledge_base(query):
    kb = {
        "서울 인구": "서울특별시의 인구는 약 950만 명입니다 (2024년 기준).",
        "서울 면적": "서울특별시의 면적은 약 605.2 km2 입니다.",
        "한국 GDP": "대한민국의 GDP는 약 1조 7천억 달러입니다 (2024년 기준).",
        "한국 인구": "대한민국의 총 인구는 약 5,170만 명입니다 (2024년 기준).",
        "파이썬": "Python은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다.",
        "인공지능": "인공지능(AI)은 인간의 지능을 모방하는 컴퓨터 시스템을 연구하는 분야입니다."
    }
    query_lower = query.lower()
    for key, value in kb.items():
        if key in query_lower or all(word in query_lower for word in key.split()):
            return value
    return f"'{query}'에 대한 정보를 찾을 수 없습니다."

In [47]:
agent = SimpleReactAgent()
agent.register_tool('Calc',calculator,'수학 수식을 계산합니다. 예 Calc[2+3]')
agent.register_tool('knowbs',knowledge_base,'내장 지식베이스에서 정보를 검색합니다. 예 knowbs[서울 인구]')
result = agent.run('서울의 인구와 면적을 조회하고, 인구밀도(인구/면적)을 계산해주세요')

Question:서울의 인구와 면적을 조회하고, 인구밀도(인구/면적)을 계산해주세요

 -- Step 1 --
Thought: 서울의 인구와 면적을 먼저 조회한 뒤, 인구밀도를 계산해야 합니다. 먼저 인구와 면적 정보를 검색합니다.
Action: knowbs[서울 인구 면적]
Thought: 서울의 인구와 면적을 먼저 조회한 뒤, 인구밀도를 계산해야 합니다. 먼저 인구와 면적 정보를 검색합니다.
Action: knowbs[서울 인구 면적]

Observation: 서울특별시의 인구는 약 950만 명입니다 (2024년 기준).

 -- Step 2 --
Thought: 인구는 확인되었고, 면적을 추가로 조회한 뒤 인구밀도를 계산해야 합니다. 먼저 서울의 면적을 검색합니다.
Action: knowbs[서울 면적]
Thought: 인구는 확인되었고, 면적을 추가로 조회한 뒤 인구밀도를 계산해야 합니다. 먼저 서울의 면적을 검색합니다.  
Action: knowbs[서울 면적]

Observation: 서울특별시의 면적은 약 605.2 km2 입니다.

 -- Step 3 --
Thought: 서울 인구(약 950만 명)와 면적(약 605.2 km²)을 확보했으므로, 인구밀도 = 인구/면적을 계산해 최종 답을 제시한다.
Action: Calc[9500000/605.2]

Observation: 15697.290152015861

 -- Step 4 --
Thought: 인구(약 950만 명)와 면적(약 605.2 km²)을 이용해 인구밀도(인구/면적)를 계산했으므로, 이를 정리해 최종 답을 제공한다.
Action: Finish[서울특별시 인구(약 950만 명)와 면적(약 605.2 km²) 기준 인구밀도는 약 15,697명/km² 입니다.]

final Answer : 서울특별시 인구(약 950만 명)와 면적(약 605.2 km²) 기준 인구밀도는 약 15,697명/km² 입니다.


# [7교시]

## 1. OpenAI Function Calling의 개념

### 텍스트 기반 ReAct vs Function Calling

이전에 구현한 텍스트 기반 ReAct 에이전트는 LLM의 출력을 직접 파싱하여 어떤 도구를 호출할지 결정했다. 이 방식은 유연하지만, 문자열 파싱에 의존하기 때문에 오류가 발생할 가능성이 높다.

OpenAI의 **Function Calling**은 이 문제를 구조적으로 해결한다. LLM이 자유 텍스트 대신 **구조화된 JSON 형태**로 함수 호출 정보를 반환하므로, 별도의 파싱 로직 없이도 안정적으로 도구를 사용할 수 있다.

### Function Calling의 주요 장점

| 장점 | 설명 |
|------|------|
| 구조화된 출력 | JSON 형태로 함수명과 인자를 반환하여 파싱 오류를 방지 |
| 타입 안전성 | JSON Schema로 인자의 타입과 제약조건을 정의 가능 |
| 병렬 호출 | 여러 함수를 동시에 호출할 수 있는 Parallel Function Calling 지원 |
| 선택적 제어 | tool_choice 파라미터로 도구 호출 방식을 세밀하게 제어 |

### tools 파라미터와 JSON Schema

Function Calling을 사용하려면 API 호출 시 `tools` 파라미터에 사용 가능한 함수들의 스키마를 전달한다. 각 함수는 다음과 같은 JSON Schema 형식으로 정의된다:

```python
tools = [{
    "type": "function",
    "function": {
        "name": "함수명",
        "description": "함수 설명",
        "parameters": {
            "type": "object",
            "properties": {
                "param1": {"type": "string", "description": "설명"}
            },
            "required": ["param1"]
        }
    }
}]
```

### tool_choice 옵션

| 옵션 | 동작 |
|------|------|
| `"auto"` | 모델이 자동으로 함수 호출 여부를 결정 (기본값) |
| `"required"` | 반드시 하나 이상의 함수를 호출하도록 강제 |
| `"none"` | 함수 호출을 하지 않고 텍스트만 생성 |
| `{"type": "function", "function": {"name": "함수명"}}` | 특정 함수를 반드시 호출 |

In [55]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY') )
# JSON Schema를 사용하여 도구 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름 (예: Seoul, Tokyo)"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "온도 단위"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "수학 계산을 수행합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 수식 (예: 2 + 3 * 4)"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]
# 도구 구현 함수
from numpy import sqrt

import requests
def get_weather(city,unit='celsius'):
    '''wttr.in 공개 api를 사용해서 실제 날씨를 조회'''
    try:
        url = f'https://wttr.in/{city}?format=j1'
        resp = requests.get(url,timeout=10)
        data = resp.json()
        current = data['current_condition'][0]
        temp_c = current['temp_C']
        temp_f = current['temp_F']
        desc = current['weatherDesc'][0]['value']
        humidity = current['humidity']
        temp = temp_c if unit=='celsius' else temp_f
        unit_str = 'C' if unit=='celsius' else 'F'
        return json.dumps({
            'city':city,
            'temperature' : f'{temp} degreees {unit_str}',
            'description' : desc,
            'humidity' : f'{humidity}%'
        },ensure_ascii=False)
    except Exception as e:
        return json.dumps({'city':city, 'temperature' : '22 degrees C'})

def calculate(expression):
    """수학 수식을 안전하게 계산"""
    try:
        result = eval(expression)
        return json.dumps({"expression": expression, "result": str(result)})
    except Exception as e:
        return json.dumps({"error": str(e)})
# 함수명으로 실제 함수를 매핑하는 dictionary  이전 코드에서 tools 역활
available_functions = {
    'get_weather':get_weather,
    'calculate' : calculate
}         

# Function calling을 포함한 api호출
response = client.chat.completions.create(
    model='gpt-5.4-nano',
    messages=[{'role':'user','content':'root(((2026-2020)33 -20)**2)'}],
    tools=tools,
    tool_choice='auto',
    temperature=0
)
print('======= Funtion Calling 응답 분석 =========')
message = response.choices[0].message
print(f'Content : {message.content}')
print(f'Tool Calls : {message.tool_calls}')
if message.tool_calls:
    for tool_call in message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        print(f'\n호출된 함수 : {func_name}')
        print(f'인자 : {func_args}')
        # 함수실행
        result = available_functions[func_name](**func_args)
        print(f'실행 결과 : {result}')

======= Funtion Calling 응답 분석 =========
Content : None
Tool Calls : [ChatCompletionMessageFunctionToolCall(id='call_QLZ85wF4RODrqhXsOx0pvEpl', function=Function(arguments='{"expression":"sqrt(((2026-2020)*33 -20)^2)"}', name='calculate'), type='function')]

호출된 함수 : calculate
인자 : {'expression': 'sqrt(((2026-2020)*33 -20)^2)'}
실행 결과 : {"expression": "sqrt(((2026-2020)*33 -20)^2)", "result": "13.2664991614216"}


# [8교시]

## 3. Function Calling 기반 ReAct 루프

Function Calling을 사용한 ReAct 루프는 텍스트 기반 방식보다 훨씬 구조적이다. 전체 흐름은 다음과 같다:

1. 사용자 질문을 메시지에 추가하고 API 호출
2. 응답에 `tool_calls`가 있으면 해당 함수를 실행
3. 실행 결과를 `role: tool` 메시지로 추가하고 다시 API 호출
4. `tool_calls`가 없으면 (최종 텍스트 응답) 루프 종료

이 과정에서 텍스트 파싱이 전혀 필요하지 않다는 것이 핵심적인 차이점이다.

In [56]:
def run_function_calling_agent(question, tools, available_functions, max_steps=5):
    """Function Calling 기반 ReAct 에이전트 루프"""
    messages = [{"role": "user", "content": question}]

    print(f"Question: {question}")
    print("=" * 60)

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-5.4-nano-2026-03-17",
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            print(f"\n--- Step {step + 1}: Tool Calls ---")
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                print(f"  Function: {func_name}({func_args})")

                result = available_functions[func_name](**func_args)
                print(f"  Result: {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        else:
            # tool_calls가 없으면 최종 응답
            print(f"\n--- Final Response ---")
            print(message.content)
            return message.content

    return "최대 단계 도달"

# 다단계 질문으로 테스트: 날씨 비교 + 온도 차이 계산
result = run_function_calling_agent(
    "서울과 도쿄의 현재 날씨를 비교해주세요. 두 도시의 온도 차이도 계산해주세요.",
    tools,
    available_functions
)

Question: 서울과 도쿄의 현재 날씨를 비교해주세요. 두 도시의 온도 차이도 계산해주세요.

--- Step 1: Tool Calls ---
  Function: get_weather({'city': 'Seoul', 'unit': 'celsius'})
  Result: {"city": "Seoul", "temperature": "25 degreees C", "description": "Partly cloudy", "humidity": "79%"}
  Function: get_weather({'city': 'Tokyo', 'unit': 'celsius'})
  Result: {"city": "Tokyo", "temperature": "22 degreees C", "description": "Partly cloudy", "humidity": "100%"}

--- Step 2: Tool Calls ---
  Function: calculate({'expression': '25-22'})
  Result: {"expression": "25-22", "result": "3"}

--- Final Response ---
현재 날씨 비교(섭씨 기준):

- **서울**: **25°C**, 부분적으로 흐림, 습도 **79%**
- **도쿄**: **22°C**, 부분적으로 흐림, 습도 **100%**

**온도 차이**: **서울 - 도쿄 = 3°C** (서울이 더 따뜻합니다)


## 4. Parallel Function Calling

Parallel Function Calling은 모델이 단일 턴에서 여러 개의 함수를 동시에 호출할 수 있는 기능이다. 바로 이 기능의 예시이다.

### 단일 호출 vs 병렬 호출

| 방식 | 동작 | 효율성 |
|------|------|--------|
| 단일 호출 | 한 번에 하나의 함수만 호출, 결과 확인 후 다음 호출 | API 호출 횟수 증가 |
| 병렬 호출 | 독립적인 여러 함수를 한 턴에서 동시 호출 | API 호출 횟수 감소, 지연 시간 단축 |

모델은 여러 함수 호출이 서로 독립적일 때 자동으로 병렬 호출을 수행한다. 예를 들어 세 도시의 날씨를 동시에 조회하는 경우가 해당된다.

In [59]:
# parallel function calling :  4개 도시 동시 조회
response = client.chat.completions.create(
    model = 'gpt-5.4-nano-2026-03-17',
    messages=[{'role':'user','content':'서울, 도쿄,베이징,뉴욕의 날씨를 모두 알려주세요'}],
    tools=tools,
    tool_choice="auto",
    temperature=0
)
message = response.choices[0].message
print(f'동시 호출된 함수 수 : {len(message.tool_calls) if message.tool_calls else 0}')
if message.tool_calls:
    for tool_call in message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        print(f'function name : {func_name}')
        print(f'func_args name : {func_args}')
        result = available_functions[func_name](**func_args)
        print(f'result : {result}\n')

동시 호출된 함수 수 : 4
function name : get_weather
func_args name : {'city': 'Seoul', 'unit': 'celsius'}
result : {"city": "Seoul", "temperature": "25 degreees C", "description": "Partly cloudy", "humidity": "79%"}

function name : get_weather
func_args name : {'city': 'Tokyo', 'unit': 'celsius'}
result : {"city": "Tokyo", "temperature": "22 degreees C", "description": "Partly cloudy", "humidity": "100%"}

function name : get_weather
func_args name : {'city': 'Beijing', 'unit': 'celsius'}
result : {"city": "Beijing", "temperature": "29 degreees C", "description": "Sunny", "humidity": "29%"}

function name : get_weather
func_args name : {'city': 'New York', 'unit': 'celsius'}
result : {"city": "New York", "temperature": "21 degreees C", "description": "Clear ", "humidity": "46%"}



## 5. 실제 API 연동 - 환율 계산

이번에는 실제 환율 API(open.er-api.com)를 연동하여 보다 실용적인 도구를 추가한다. 환율 조회와 수학 계산을 결합한 복합 질문으로 Function Calling의 다단계 추론 능력을 확인한다.

- https://open.er-api.com/v6/latest/usd

In [60]:
# 환율 조회 도구 추가
def get_exchange_rate(base_currency, target_currency):
    """open.er-api.com을 사용하여 실시간 환율을 조회"""
    import requests
    try:
        url = f"https://open.er-api.com/v6/latest/{base_currency}"
        resp = requests.get(url, timeout=10)
        data = resp.json()
        rate = data['rates'].get(target_currency)
        if rate:
            return json.dumps({"base": base_currency, "target": target_currency, "rate": rate})
        return json.dumps({"error": f"{target_currency} not found"})
    except Exception as e:
        return json.dumps({"error": str(e)})

# 환율 도구의 JSON Schema 정의
exchange_tool = {
    "type": "function",
    "function": {
        "name": "get_exchange_rate",
        "description": "두 통화 간의 환율을 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "base_currency": {"type": "string", "description": "기준 통화 (예: USD, KRW, JPY)"},
                "target_currency": {"type": "string", "description": "대상 통화 (예: USD, KRW, JPY)"}
            },
            "required": ["base_currency", "target_currency"]
        }
    }
}

# 모든 도구를 통합
all_tools = tools + [exchange_tool]
available_functions["get_exchange_rate"] = get_exchange_rate

# 환율 조회 + 계산이 결합된 복합 질문 실행
result = run_function_calling_agent(
    "현재 1달러는 몇 원인지 조회하고, 350달러를 원화로 환산해주세요.",
    all_tools,
    available_functions
)

Question: 현재 1달러는 몇 원인지 조회하고, 350달러를 원화로 환산해주세요.

--- Step 1: Tool Calls ---
  Function: get_exchange_rate({'base_currency': 'USD', 'target_currency': 'KRW'})
  Result: {"base": "USD", "target": "KRW", "rate": 1500.684943}

--- Final Response ---
현재 환율 기준으로 **1달러(USD) = 1,500.684943원(KRW)** 입니다.

따라서 **350달러 = 350 × 1,500.684943 = 525,239.73005원** 으로 환산됩니다.
